In [12]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import os
plt.style.use('default')
plt.rcParams['font.sans-serif']=['SimHei']
plt.rcParams['axes.unicode_minus']=False 
import pickle
import tensorflow as tf
from sklearn.preprocessing import StandardScaler
from tensorflow.keras.models import Model
from sklearn.preprocessing import StandardScaler
from tensorflow.keras.layers import Conv1D,Conv2D,MaxPooling1D, MaxPooling2D, Flatten, Dense, Input, Dropout, LSTM, TimeDistributed, Reshape, BatchNormalization, Activation, concatenate
cm = 1/2.54
# 字号
fontsize = 8
seed = 2025

In [13]:
# 加载数据集，Ctrain_CC表示曲线训练集的CC数据，Htrain_CC表示曲线训练集的HI数据
with open(r'G:\boshi\1_boyi\3_ziji\My_program\long\3\code\5-fusion\data_CC.pkl', 'rb') as file:
    Ctrain_CC, Cval_CC, Ctest_CC, Htrain_CC, Hval_CC, Htest_CC = pickle.load(file)
    file.close()
with open(r'G:\boshi\1_boyi\3_ziji\My_program\long\3\code\5-fusion\data_CV.pkl', 'rb') as file:
    Ctrain_CV, Cval_CV, Ctest_CV, Htrain_CV, Hval_CV, Htest_CV = pickle.load(file)
    file.close()
with open(r'G:\boshi\1_boyi\3_ziji\My_program\long\3\code\5-fusion\data_RE.pkl','rb') as file:
    Ctrain_RE, Cval_RE, Ctest_RE, Htrain_RE, Hval_RE, Htest_RE = pickle.load(file)
    file.close()

In [14]:
Htrain_CC

,CC_Kur,CC_Min,CC_Mean,cycle,capacity,id
488,-0.866229,4.117173,4.166539,0.0,3269.007095,25-025_1-#2
489,-0.859623,4.116938,4.166001,1.0,3269.019540,25-025_1-#2
490,-0.841131,4.119564,4.167292,2.0,3267.816991,25-025_1-#2
491,-0.837301,4.119723,4.16726,3.0,3265.782539,25-025_1-#2
492,-0.825544,4.12013,4.167336,4.0,3263.622593,25-025_1-#2
...,...,...,...,...,...,...
22245,-1.199066,3.966743,4.07203,395.0,2833.534224,45-05_1-#9
22246,-1.200981,3.966932,4.072354,396.0,2832.233180,45-05_1-#9
22247,-1.202707,3.967177,4.072592,397.0,2832.148549,45-05_1-#9
22248,-1.170782,3.963421,4.068223,398.0,2831.231334,45-05_1-#9


### 早期融合：直接拼接曲线和HI

#### 模型

In [15]:
def CNN_model(input_shape):
    input_layer = Input(shape=input_shape)
    x = Conv1D(filters=16, kernel_size=5, strides=5, activation='relu')(input_layer)
    x = MaxPooling1D(pool_size=2, strides=2)(x)
    x = Conv1D(filters=32, kernel_size=5, strides=2, activation='relu')(x)
    x = Conv1D(filters=32, kernel_size=3, strides=1, activation='relu')(x)
    x = Conv1D(filters=64, kernel_size=3, strides=1, activation='relu')(x)
    x = MaxPooling1D(pool_size=2, strides=1)(x)
    x = Dense(64, activation='relu')(x)
    x = Dropout(0.1)(x)
    x = Dense(32, activation='relu')(x)
    x = Dropout(0.1)(x)
    x = Dense(32, activation='relu')(x)
    x = Dropout(0.1)(x)
    x = Dense(1, activation='relu')(x)
    model = Model(inputs=input_layer, outputs=x)
    model.compile(optimizer='adam', loss='mse')
    return model
def model_train(model, train_x, train_y, val_x, val_y, epochs):
    train_y = train_y.iloc[:, -2]/3500 # 计算SOH
    val_y = val_y.iloc[:, -2]/3500
    train_x = train_x.reshape(train_x.shape[0], train_x.shape[1], 1) # 修改形状
    val_x = val_x.reshape(val_x.shape[0], val_x.shape[1], 1)
    model.fit(train_x, train_y, epochs=epochs, validation_data=(val_x, val_y), verbose=0)
    return model
def model_eval(model, test_x, test_y):
    x = test_x.reshape(test_x.shape[0], test_x.shape[1], 1)
    y_pred = model.predict(x)
    # 修改形状
    y_pred = y_pred.reshape(y_pred.shape[0])
    y_pred = pd.DataFrame(y_pred, columns=['pred'])
    # 将test_y的capacity列转变为SOH
    test_y['SOH'] = test_y['capacity']/3500
    test_y.reset_index(drop=True, inplace=True)
    result = pd.concat([test_y, y_pred], axis=1)
    return result

#### CC阶段

In [16]:
def get_inut(Curve, HI):
    x = pd.concat([Curve.iloc[:, :-3], HI.iloc[:, :-3]], axis=1)
    y = HI.iloc[:, -3:]
    return x, y

In [17]:
train_x, train_y = get_inut(Ctrain_CC, Htrain_CC)
val_x, val_y = get_inut(Cval_CC, Hval_CC)
test_x, test_y = get_inut(Ctest_CC, Htest_CC)
# 标准化
scaler = StandardScaler()
train_x = scaler.fit_transform(train_x)
val_x = scaler.transform(val_x)
test_x = scaler.transform(test_x)

In [18]:
model_CC = CNN_model((train_x.shape[1], 1))
# 可视化模型结构
model_CC.summary()

Model: "functional"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ input_layer (InputLayer)        │ (None, 154, 1)         │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv1d (Conv1D)                 │ (None, 30, 16)         │            96 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling1d (MaxPooling1D)    │ (None, 15, 16)         │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv1d_1 (Conv1D)               │ (None, 6, 32)          │         2,592 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv1d_2 (Conv1D)               │ (None, 4, 32)          │         3,104 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv1d_3 (Conv1D)               │ (None, 2, 64)          │         6,208 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling1d_1 (MaxPooling1D)  │ (None, 1, 64)          │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 1, 64)          │         4,160 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 1, 64)          │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 1, 32)          │         2,080 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_1 (Dropout)             │ (None, 1, 32)          │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ (None, 1, 32)          │         1,056 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_2 (Dropout)             │ (None, 1, 32)          │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_3 (Dense)                 │ (None, 1, 1)           │            33 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 19,329 (75.50 KB)

 Trainable params: 19,329 (75.50 KB)

 Non-trainable params: 0 (0.00 B)

In [ ]:
# CC
epochs = 100
model_CC = model_train(model_CC, train_x, train_y, val_x, val_y, epochs)
# 保存模型
model_CC.save(r'G:\boshi\1_boyi\3_ziji\My_program\long\3\code\5-fusion\model\early\model_CC.h5')
result_CC = model_eval(model_CC, test_x, test_y)
result_CC.to_csv(r'G:\boshi\1_boyi\3_ziji\My_program\long\3\code\5-fusion\rusult\early\result_CC.csv', index=False)


#### CV阶段

In [ ]:
train_x, train_y = get_inut(Ctrain_CV, Htrain_CV)
val_x, val_y = get_inut(Cval_CV, Hval_CV)
test_x, test_y = get_inut(Ctest_CV, Htest_CV)
# 标准化
scaler = StandardScaler()
train_x = scaler.fit_transform(train_x)
val_x = scaler.transform(val_x)
test_x = scaler.transform(test_x)

In [ ]:
model_CV = CNN_model((train_x.shape[1], 1))
epochs = 100
model_CV = model_train(model_CV, train_x, train_y, val_x, val_y, epochs)
# 保存模型
model_CV.save(r'G:\boshi\1_boyi\3_ziji\My_program\long\3\code\5-fusion\model\early\model_CV.h5')
result_CV = model_eval(model_CV, test_x, test_y)
result_CV.to_csv(r'G:\boshi\1_boyi\3_ziji\My_program\long\3\code\5-fusion\rusult\early\result_CV.csv', index=False)


#### RE阶段

In [ ]:
train_x, train_y = get_inut(Ctrain_RE, Htrain_RE)
val_x, val_y = get_inut(Ctrain_RE, Htrain_RE)
test_x, test_y = get_inut(Ctrain_RE, Htrain_RE)
# 标准化
scaler = StandardScaler()
train_x = scaler.fit_transform(train_x)
val_x = scaler.transform(val_x)
test_x = scaler.transform(test_x)

In [ ]:
model_RE = CNN_model((train_x.shape[1], 1))
epochs = 100
model_RE = model_train(model_RE, train_x, train_y, val_x, val_y, epochs)
# 保存模型
model_RE.save(r'G:\boshi\1_boyi\3_ziji\My_program\long\3\code\5-fusion\model\early\model_RE.h5')
result_RE = model_eval(model_RE, test_x, test_y)
result_RE.to_csv(r'G:\boshi\1_boyi\3_ziji\My_program\long\3\code\5-fusion\rusult\early\result_RE.csv', index=False)
